# Lab 08 · Reference solution

The polished final implementation of [Lab 08: Contextual retrieval and query rewriting](../README.md).

Composes Lab 07's hybrid+rerank pipeline with two upstream layers:

- **Contextual augmentation** — Anthropic's verbatim contextual-retrieval
  prompt generates a 1-2 sentence summary per chunk that gets prepended
  to the chunk *before indexing*. Cached on disk; regenerated only for
  new/missing entries.
- **Query rewriting** — three modes:
  - `hyde` — generate a hypothetical answer; embed the answer as query.
  - `multi` — generate 3 rephrasings; retrieve against each; RRF-fuse.
  - `decompose` — break compound queries into atomic sub-queries.

Composed as `search_corpus_v3` with the same tool contract Labs 06/07
exposed, so the agent loop is unchanged.

> ⏱ Read time: ~12 min · Notebook ~25 cells.
> 📖 The lab walks each layer through targeted failure modes — read it
> for *when* each rewrite mode helps and when it hurts.

> 🔒 **Chunker config pinned**: `TARGET_TOKENS=160`, `OVERLAP_TOKENS=32`.
> Don't change — Lab 09's eval set annotates against these chunk IDs.

> 💾 **Cache path is `../context_cache.json`** — one level up from this
> solution dir. If you've already run the lab notebook, that cache is
> reused here verbatim (same chunk IDs, same context summaries). No
> wasted LLM calls.

## Setup

In [ ]:
import hashlib
import json
import os
import pathlib
import re
from typing import Any

from dotenv import load_dotenv

here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / ".env.example").exists():
        load_dotenv(parent / ".env")
        break

assert os.getenv("OPENAI_API_KEY") or os.getenv("ANTHROPIC_API_KEY")

PROVIDER = "openai"
MODEL = {
    "openai": "gpt-4o-mini",
    "anthropic": "claude-haiku-4-5-20251001",
}[PROVIDER]
print(f"Using {PROVIDER} / {MODEL}")


## Corpus + chunker

Same as Labs 06 and 07. The full `docs` dict (whole-document text by
filename) is needed by the contextualizer prompt.

In [ ]:
CORPUS_DIR = pathlib.Path("../../06-agentic-rag-from-scratch/corpus")
TARGET_TOKENS = 160
OVERLAP_TOKENS = 32


def approx_tokens(text: str) -> int:
    return int(len(text.split()) / 0.75)


def split_at_paragraphs(text: str) -> list[str]:
    return [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]


def split_at_sentences(text: str) -> list[str]:
    return [p.strip() for p in re.split(r"(?<=[.!?])\s+", text) if p.strip()]


def chunk_text(text: str) -> list[str]:
    paragraphs = split_at_paragraphs(text)
    chunks: list[str] = []
    current: list[str] = []
    current_tokens = 0
    for para in paragraphs:
        para_tokens = approx_tokens(para)
        if para_tokens > TARGET_TOKENS:
            if current:
                chunks.append("\n\n".join(current))
                current, current_tokens = [], 0
            sentences = split_at_sentences(para)
            sub_chunk: list[str] = []
            sub_tokens = 0
            for sent in sentences:
                sent_tokens = approx_tokens(sent)
                if sub_tokens + sent_tokens > TARGET_TOKENS and sub_chunk:
                    chunks.append(" ".join(sub_chunk))
                    sub_chunk, sub_tokens = [], 0
                sub_chunk.append(sent)
                sub_tokens += sent_tokens
            if sub_chunk:
                chunks.append(" ".join(sub_chunk))
            continue
        if current_tokens + para_tokens > TARGET_TOKENS and current:
            chunks.append("\n\n".join(current))
            current, current_tokens = [], 0
        current.append(para)
        current_tokens += para_tokens
    if current:
        chunks.append("\n\n".join(current))
    if OVERLAP_TOKENS <= 0 or len(chunks) < 2:
        return chunks
    overlapped: list[str] = [chunks[0]]
    for i in range(1, len(chunks)):
        prev_words = chunks[i - 1].split()
        overlap_words = int(OVERLAP_TOKENS * 0.75)
        tail = " ".join(prev_words[-overlap_words:]) if overlap_words > 0 else ""
        overlapped.append((tail + " " + chunks[i]).strip() if tail else chunks[i])
    return overlapped


def first_heading(text: str) -> str:
    for line in text.splitlines():
        if line.startswith("# "):
            return line[2:].strip()
    return ""


docs: dict[str, str] = {}
all_chunks: list[dict] = []
for path in sorted(CORPUS_DIR.glob("*.md")):
    if path.name == "README.md":
        continue
    text = path.read_text()
    docs[path.name] = text
    title = first_heading(text)
    for i, chunk_body in enumerate(chunk_text(text)):
        all_chunks.append({
            "chunk_id": f"{path.name}:{i}",
            "doc_id": path.name,
            "title": title,
            "text": chunk_body,
        })

chunks_by_id = {c["chunk_id"]: c for c in all_chunks}
print(f"Loaded {len(docs)} docs, {len(all_chunks)} chunks")


## Provider-agnostic LLM client

`llm_complete` for single-turn generation (used by the contextualizer
and the query-rewriting prompts). `chat_with_tools` for the agent loop.

In [ ]:
def llm_complete(prompt: str, max_tokens: int = 256) -> str:
    """Single-turn completion. Returns the model's text response."""
    if PROVIDER == "openai":
        from openai import OpenAI
        resp = OpenAI().chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens, temperature=0,
        )
        return resp.choices[0].message.content or ""
    elif PROVIDER == "anthropic":
        from anthropic import Anthropic
        resp = Anthropic().messages.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens,
        )
        return "".join(b.text for b in resp.content if hasattr(b, "text"))
    raise ValueError(f"Unknown PROVIDER: {PROVIDER!r}")


def chat_with_tools(messages: list[dict], tools: list[dict]) -> dict:
    """Multi-turn with tools — same as Labs 06/07."""
    if PROVIDER == "openai":
        from openai import OpenAI
        resp = OpenAI().chat.completions.create(
            model=MODEL, messages=messages, tools=tools, temperature=0,
        )
        msg = resp.choices[0].message
        return {
            "role": "assistant", "content": msg.content or "",
            "tool_calls": [
                {"id": tc.id, "name": tc.function.name, "arguments": tc.function.arguments}
                for tc in (msg.tool_calls or [])
            ],
        }
    elif PROVIDER == "anthropic":
        from anthropic import Anthropic
        client = Anthropic()
        anth_tools = [
            {"name": t["function"]["name"],
             "description": t["function"]["description"],
             "input_schema": t["function"]["parameters"]}
            for t in tools
        ]
        system = next((m["content"] for m in messages if m["role"] == "system"), "")
        non_system = [m for m in messages if m["role"] != "system"]
        resp = client.messages.create(
            model=MODEL, system=system, messages=non_system,
            tools=anth_tools, max_tokens=2048, temperature=0,
        )
        text = "".join(b.text for b in resp.content if hasattr(b, "text"))
        tool_calls = [
            {"id": b.id, "name": b.name, "arguments": json.dumps(b.input)}
            for b in resp.content if getattr(b, "type", None) == "tool_use"
        ]
        return {"role": "assistant", "content": text, "tool_calls": tool_calls}
    raise ValueError(f"Unknown PROVIDER: {PROVIDER!r}")


## Contextualizer (Anthropic's verbatim prompt)

The `CONTEXT_PROMPT` template is reproduced verbatim from Anthropic's
[contextual retrieval announcement](https://www.anthropic.com/news/contextual-retrieval).
The cache-or-generate pattern means an unchanged corpus generates no
new LLM calls.

In [ ]:
CONTEXT_PROMPT = """<document>
{whole_document}
</document>

Here is the chunk we want to situate within the whole document
<chunk>
{chunk_content}
</chunk>

Please give a short succinct context to situate this chunk within the overall document for the purposes of improving search retrieval of the chunk. Answer only with the succinct context and nothing else."""

CACHE_PATH = pathlib.Path("../context_cache.json")


def load_context_cache() -> dict[str, str]:
    if CACHE_PATH.exists():
        return json.loads(CACHE_PATH.read_text())
    return {}


def save_context_cache(cache: dict[str, str]) -> None:
    CACHE_PATH.write_text(json.dumps(cache, indent=2))


def contextualize_chunk(chunk: dict, cache: dict[str, str]) -> str:
    """Return context summary; generates only on cache miss."""
    if chunk["chunk_id"] in cache:
        return cache[chunk["chunk_id"]]
    prompt = CONTEXT_PROMPT.format(
        whole_document=docs[chunk["doc_id"]],
        chunk_content=chunk["text"],
    )
    context = llm_complete(prompt, max_tokens=180).strip()
    cache[chunk["chunk_id"]] = context
    return context


cache = load_context_cache()
new_calls = 0
print("Generating context summaries (cached entries reused)...")
for chunk in all_chunks:
    if chunk["chunk_id"] not in cache:
        contextualize_chunk(chunk, cache)
        new_calls += 1

save_context_cache(cache)
print(f"  {new_calls} new LLM calls; {len(cache) - new_calls} loaded from cache.")
print(f"Total chunks with context: {len(cache)}/{len(all_chunks)}")


**Sample output (first run; subsequent runs are 0 new calls):**

```
Generating context summaries (cached entries reused)...
  55 new LLM calls; 0 loaded from cache.
Total chunks with context: 55/55
```

The cache lives at `../context_cache.json`. A subsequent run with the
same corpus produces no new LLM calls.

## Contextual indexes

The augmented text is `f"{context}\n\n{chunk_text}"`. We index the
augmented form; we *return* the original chunk text at lookup time (the
agent reads the original — the context summary was only for retrieval).

In [ ]:
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

print("Loading bi-encoder...")
embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cpu",
)


def tokenize(text: str) -> list[str]:
    return [t for t in re.findall(r"\w+", text.lower()) if len(t) > 1]


augmented_texts = [
    f"{cache[c['chunk_id']]}\n\n{c['text']}"
    for c in all_chunks
]

emb_contextual = embedder.encode(
    augmented_texts, normalize_embeddings=True,
    convert_to_numpy=True, show_progress_bar=False,
)
bm25_contextual = BM25Okapi([tokenize(t) for t in augmented_texts])

baseline_avg = np.mean([len(tokenize(c["text"])) for c in all_chunks])
augmented_avg = np.mean([len(tokenize(t)) for t in augmented_texts])
print(f"Contextual indexes: dense={emb_contextual.shape}, bm25 over {len(augmented_texts)} chunks")
print(f"Avg tokens: baseline={baseline_avg:.1f}, contextual={augmented_avg:.1f} "
      f"(+{(augmented_avg/baseline_avg - 1) * 100:.0f}%)")


## Hybrid retrieval over the contextual indexes

Same RRF + cross-encoder rerank as Lab 07; just pointed at the
contextual indexes.

In [ ]:
def dense_retrieve(emb_index: np.ndarray, query: str, top_k: int = 10) -> list[tuple[int, float]]:
    q = embedder.encode([query], normalize_embeddings=True,
                        convert_to_numpy=True, show_progress_bar=False)[0]
    scores = emb_index @ q
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [(int(i), float(scores[i])) for i in top_indices]


def bm25_retrieve(bm25: BM25Okapi, query: str, top_k: int = 10) -> list[tuple[int, float]]:
    q_tokens = tokenize(query)
    scores = bm25.get_scores(q_tokens)
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [(int(i), float(scores[i])) for i in top_indices]


def reciprocal_rank_fusion(
    ranked_lists: dict[str, list[tuple[int, float]]],
    k: int = 60,
) -> list[tuple[int, float]]:
    rrf_scores: dict[int, float] = {}
    for ranked in ranked_lists.values():
        for rank, (chunk_idx, _score) in enumerate(ranked, start=1):
            rrf_scores[chunk_idx] = rrf_scores.get(chunk_idx, 0.0) + 1.0 / (k + rank)
    return sorted(rrf_scores.items(), key=lambda kv: kv[1], reverse=True)


def hybrid_retrieve(emb_index, bm25, query: str,
                    top_k: int = 10, candidate_k: int = 30) -> list[tuple[int, float]]:
    d = dense_retrieve(emb_index, query, top_k=candidate_k)
    b = bm25_retrieve(bm25, query, top_k=candidate_k)
    fused = reciprocal_rank_fusion({"dense": d, "bm25": b}, k=60)
    return fused[:top_k]


## Query rewriting — three modes

**HyDE** (Hypothetical Document Embeddings). Gao et al. 2022. Generate
a plausible answer; embed the *answer*, not the query. Closes the
question-answer asymmetry in embedding space.

**Multi-query expansion**. Generate 3 paraphrasings; retrieve against
each; RRF-fuse. Robust to vocabulary mismatch.

**Decomposition**. Break compound questions ("X and what about Y?") into
atomic sub-queries. Retrieve against each.

In [ ]:
HYDE_PROMPT = """Write a 2-3 sentence passage that could answer the following question. Be specific and use technical vocabulary the way an expert would; the passage doesn't need to be factually correct, only plausibly worded.

Question: {query}

Passage:"""


MULTI_QUERY_PROMPT = """Generate 3 different ways to phrase the following question for use in a search engine. Use distinct vocabulary and phrasing in each. Return only the 3 phrasings, one per line, with no numbering or explanations.

Question: {query}"""


DECOMPOSE_PROMPT = """Decompose the following compound question into a list of atomic sub-questions, one per line. Return only the sub-questions, no numbering or explanations. If the question is already atomic, return just the original.

Question: {query}"""


def hyde_rewrite(query: str) -> list[str]:
    """Returns [hypothetical_answer] — single-element list for uniform API."""
    return [llm_complete(HYDE_PROMPT.format(query=query), max_tokens=200).strip()]


def multi_query_rewrite(query: str) -> list[str]:
    """Returns [original, rephrase_1, rephrase_2, rephrase_3]."""
    output = llm_complete(MULTI_QUERY_PROMPT.format(query=query), max_tokens=200).strip()
    rephrasings = [
        line.strip().lstrip("-*0123456789.) ").strip()
        for line in output.split("\n")
        if line.strip()
    ]
    return [query, *rephrasings[:3]]


def decompose_query(query: str) -> list[str]:
    """Returns [sub_question_1, sub_question_2, ...]."""
    output = llm_complete(DECOMPOSE_PROMPT.format(query=query), max_tokens=200).strip()
    subs = [
        line.strip().lstrip("-*0123456789.) ").strip()
        for line in output.split("\n")
        if line.strip()
    ]
    return subs if subs else [query]


## Cross-encoder reranker — same as Lab 07

In [ ]:
from sentence_transformers import CrossEncoder

print("Loading reranker...")
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    device="cpu",
    max_length=512,
)


def cross_encoder_rerank(
    query: str,
    candidates: list[tuple[int, float]],
    top_k: int = 5,
) -> list[tuple[int, float]]:
    if not candidates:
        return []
    pairs = [(query, all_chunks[idx]["text"]) for idx, _score in candidates]
    rerank_scores = reranker.predict(pairs, show_progress_bar=False, convert_to_numpy=True)
    rescored = list(zip([c[0] for c in candidates], rerank_scores.tolist(), strict=True))
    rescored.sort(key=lambda x: x[1], reverse=True)
    return rescored[:top_k]


## `search_corpus_v3` — the composed pipeline

The full stack:

1. Optional **query rewriting** (`hyde` / `multi` / `decompose`).
2. **Hybrid retrieve** against contextual indexes for each rewrite.
3. **RRF fuse** across all rewrites × both retrievers.
4. **Cross-encoder rerank** using the *original* query (the rewrites were
   for retrieval; the rerank should score against what the user asked).
5. Floor at `MIN_SIMILARITY=0.0` (cross-encoder logits, same as Lab 07).
6. Return chunks with *original* text (the agent reads the chunk, not
   the augmented form).

Same envelope as Labs 06/07 — `status: ok/empty/error` — so the agent
loop is unchanged.

In [ ]:
MIN_SIMILARITY = 0.0  # Cross-encoder logits, same as Lab 07


def search_corpus_v3(
    query: str,
    top_k: int = 5,
    candidate_k: int = 30,
    rewrite_mode: str | None = None,  # None | "hyde" | "multi" | "decompose"
) -> dict:
    """Contextual hybrid + query rewriting + rerank. Same envelope as Labs 06/07."""
    if not query or not query.strip():
        return {"status": "error", "kind": "other", "detail": "empty query"}

    # 1. Optional rewriting
    if rewrite_mode == "hyde":
        rewrites = hyde_rewrite(query)
    elif rewrite_mode == "multi":
        rewrites = multi_query_rewrite(query)
    elif rewrite_mode == "decompose":
        rewrites = decompose_query(query)
    else:
        rewrites = [query]

    # 2-3. Hybrid retrieve against each rewrite; RRF fuse all signals
    ranked_lists: dict[str, list[tuple[int, float]]] = {}
    for i, q in enumerate(rewrites):
        ranked_lists[f"dense_{i}"] = dense_retrieve(emb_contextual, q, top_k=candidate_k)
        ranked_lists[f"bm25_{i}"] = bm25_retrieve(bm25_contextual, q, top_k=candidate_k)
    fused = reciprocal_rank_fusion(ranked_lists, k=60)[:candidate_k]

    # 4. Rerank using ORIGINAL query (not the rewrites)
    reranked = cross_encoder_rerank(query, fused, top_k=top_k)

    # 5. Floor
    above = [(idx, s) for idx, s in reranked if s >= MIN_SIMILARITY]
    if not above:
        top_score = reranked[0][1] if reranked else float("-inf")
        return {
            "status": "empty", "query": query,
            "detail": f"no chunks crossed rerank floor (top was {top_score:.3f})",
        }

    # 6. Build envelope — return ORIGINAL chunk text
    results = []
    for idx, score in above:
        chunk = all_chunks[idx]
        snippet = chunk["text"][:200].replace("\n", " ")
        if len(chunk["text"]) > 200:
            snippet += "..."
        results.append({
            "chunk_id": chunk["chunk_id"],
            "doc_id": chunk["doc_id"],
            "title": chunk["title"],
            "snippet": snippet,
            "score": float(score),
            "retrieval_signals": {
                "rerank": float(score),
                "rewrite_mode": rewrite_mode or "none",
                "rewrites_used": len(rewrites),
            },
        })
    return {"status": "ok", "results": results}


## Wire into the agent loop

Same loop, same tool contract. Default `rewrite_mode=None` for the tool
schema — the agent doesn't have to know about rewriting. Production
systems often pick the mode adaptively based on query shape: HyDE for
underspecified queries, multi-query for vocabulary-mismatched queries,
decomposition for compound queries.

In [ ]:
def read_chunk(chunk_id: str) -> dict:
    if not chunk_id:
        return {"status": "error", "kind": "other", "detail": "empty chunk_id"}
    chunk = chunks_by_id.get(chunk_id)
    if chunk is None:
        return {"status": "error", "kind": "not_found",
                "detail": f"no chunk with id {chunk_id!r}"}
    return {
        "status": "ok",
        "chunk_id": chunk["chunk_id"], "doc_id": chunk["doc_id"],
        "title": chunk["title"], "text": chunk["text"],
    }


TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search_corpus",
            "description": (
                "Search the corpus by contextual hybrid retrieval with cross-encoder "
                "reranking. Returns top_k chunks. Phrase queries as 3-8 specific words."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "3-8 specific words."},
                    "top_k": {"type": "integer", "description": "1-10, default 5."},
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_chunk",
            "description": "Read full text of one chunk. Pass chunk_id from search_corpus.",
            "parameters": {
                "type": "object",
                "properties": {"chunk_id": {"type": "string"}},
                "required": ["chunk_id"],
            },
        },
    },
]


def execute_tool(name: str, args: dict) -> dict:
    if name == "search_corpus":
        return search_corpus_v3(query=args["query"], top_k=args.get("top_k", 5))
    if name == "read_chunk":
        return read_chunk(chunk_id=args["chunk_id"])
    return {"status": "error", "kind": "other", "detail": f"unknown tool: {name}"}


MAX_STEPS = 8

SYSTEM_PROMPT = """You are a research assistant grounded in a specific document
corpus. Answer only from the corpus.

1. search_corpus with 3-8 specific words.
2. If snippets answer the question, synthesize. Otherwise read_chunk the most
   relevant 1-2 chunks.
3. Cite chunks you read. When the corpus doesn't cover the question, say so.
"""


def _action_hash(name: str, args: dict) -> str:
    return hashlib.sha256(
        (name + "|" + json.dumps(args, sort_keys=True)).encode()
    ).hexdigest()[:16]


def run_agent(question: str, max_steps: int = MAX_STEPS, verbose: bool = True) -> dict:
    messages: list[dict] = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    citations: list[dict] = []
    seen_actions: set[str] = set()
    for step in range(1, max_steps + 1):
        if verbose:
            print(f"\n── Step {step} ──")
        msg = chat_with_tools(messages, tools=TOOLS)
        entry: dict[str, Any] = {"role": "assistant", "content": msg["content"]}
        if msg["tool_calls"]:
            entry["tool_calls"] = [
                {"id": tc["id"], "type": "function",
                 "function": {"name": tc["name"], "arguments": tc["arguments"]}}
                for tc in msg["tool_calls"]
            ]
        messages.append(entry)
        if not msg["tool_calls"]:
            if verbose:
                print(f"  ◆ FINAL: {(msg['content'] or '')[:120]}...")
            return {
                "answer": msg["content"], "citations": citations, "steps": step,
                "stopped_reason": "answer_with_citations" if citations else "answer_without_read",
            }
        for tc in msg["tool_calls"]:
            args = json.loads(tc["arguments"]) if tc["arguments"] else {}
            ah = _action_hash(tc["name"], args)
            if ah in seen_actions:
                tool_result = {"status": "error", "kind": "repeated_action",
                               "detail": f"You already called {tc['name']} with these arguments."}
                if verbose:
                    print(f"  ✗ {tc['name']}({args}) [REPEATED]")
            else:
                seen_actions.add(ah)
                tool_result = execute_tool(tc["name"], args)
                if verbose:
                    args_repr = (str(args)[:80] + "...") if len(str(args)) > 80 else str(args)
                    print(f"  → {tc['name']}({args_repr}) → {tool_result.get('status', '?')}")
                if tc["name"] == "read_chunk" and tool_result.get("status") == "ok":
                    citations.append({
                        "chunk_id": tool_result["chunk_id"],
                        "doc_id": tool_result["doc_id"],
                        "title": tool_result["title"],
                    })
            messages.append({"role": "tool", "tool_call_id": tc["id"],
                             "content": json.dumps(tool_result)[:4000]})
    return {
        "answer": f"[step cap; read {len(citations)} chunk(s)]",
        "citations": citations, "steps": max_steps, "stopped_reason": "step_cap",
    }


## Demo

A vocabulary-mismatched query — the corpus uses "ReAct pattern" and
"thoughts before tool calls"; the user phrases it differently.
Contextual indexes catch this by augmenting each chunk with a summary
that names the underlying concept.

In [ ]:
result = run_agent(
    "How does the framework make the LLM explain its reasoning before acting?"
)
print(f"\n=== Answer ===\n{result['answer']}")
print(f"\n=== Citations ({len(result['citations'])}) ===")
for c in result["citations"]:
    print(f"  • [{c['chunk_id']}] {c['title']}")


**Sample output (rerank scores will vary; trajectory should be stable):**

```
── Step 1 ──
  → search_corpus({'query': 'LLM explain reasoning before acting'}) → ok

── Step 2 ──
  → read_chunk({'chunk_id': '03-react-pattern.md:0'}) → ok

── Step 3 ──
  ◆ FINAL: The ReAct pattern prepends a brief "Thought:" reasoning step...

=== Citations (1) ===
  • [03-react-pattern.md:0] The ReAct Pattern
```

## Production readiness — out of scope here

For deployment: prompt-cache the document portion of `CONTEXT_PROMPT`
(Anthropic claims 90% cost reduction with prompt caching enabled);
batch the contextualizer calls; choose `rewrite_mode` adaptively from
query shape (a 1-call classifier or a heuristic); cache rewrite outputs
keyed on the query string; A/B test which combination of modes wins on
your eval set.

For evaluating the contributions of each layer (contextual /
multi-query / HyDE), see Lab 09.